# Transformer(5.Residual Connection + Layer Normalization) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

---

## 2. Transformer(5.Residual Connection + Layer Normalization) 언어 모델의 1차 Residual Connection + Layer Normalization 계산 과정을 공식과 함께 단계별로 계산
---
### 1) 1차 Residual Connection
	◦ Residual Connection은 입력 정보와 Masked Multi-Head Attention 출력값을 합쳐 정보 손실을 방지한다
	◦ Residual Connection 계산 공식 : 

$ [ X^\prime = Y + \text{MaskedMHA}(Y) ] $

$ - [ Y ] $ : 디코더 입력 벡터 (임베딩 + 위치 인코딩)

$ - [ \text{MaskedMHA}(Y) ] $ : 앞서 계산한 Masked Multi-Head Attention 출력

예시 값:

$ [ Y = [0.6, 1.1], \quad \text{MaskedMHA}(Y) = [0.6, 1.1] ] $

따라서:

$ [ X^\prime = [0.6, 1.1] + [0.6, 1.1] = [1.2, 2.2] ] $

---
### 2) 1차 Layer Normalization
	◦ LayerNorm은 각 벡터를 평균과 분산을 기준으로 정규화하여 학습 안정성을 확보한다
	◦ Layer Normalization 계산 공식:

$ [ \text{LayerNorm}(X^\prime) = \frac{X^\prime - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta ] $

$ - \mu : 벡터 평균 $

$ - \sigma^2 : 벡터 분산 $

$ - \epsilon : 안정성을 위한 작은값 $

$ - \gamma , \beta : 학습 가능한 파라미터(스케일, 시프트) $

예시 계산

Residual Connection 계산된 벡터 :
$ [ X^\prime = [1.2, 2.2] ] $

2-1) 평균 :
$ [ \mu = \frac{1.2 + 2.2}{2} = 1.7 ] $

2-2) 분산 :
$ [ \sigma^2 = \frac{(1.2-1.7)^2 + (2.2-1.7)^2}{2} = \frac{0.25 + 0.25}{2} = 0.25 ] $

2-3) 표준편차 :
$ [ \sqrt{\sigma^2 + \epsilon} \approx \sqrt{0.25} = 0.5 ] $

2-4) 정규화 :
$ [ \frac{X^\prime - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{[1.2-1.7, 2.2-1.7]}{0.5} = \frac{[-0.5, 0.5]}{0.5} = [-1.0, 1.0] ] $ 

3-5) 추가(현재는 필요 없음)로 여기에 학습 파라미터 [ \gamma, \beta ]를 적용 :
$ [ \text{LayerNorm}(X^\prime) = [-1.0, 1.0] \cdot \gamma + \beta ] $

---